# Lorenz-63 emulator failure-mode sweep (driver)

This notebook is a **thin driver**. All scientific code lives in the `lorenz63` package,
so there is only one copy of it. Diagnostics are pure measurement functions and never
touch matplotlib; figures are written by `lorenz63.report`, which saves *before* showing
and verifies each file is not blank.

Set `SMOKE = True` for a fast CPU run, or `False` for the full four-point sweep.

In [ ]:
# Install dependencies. On Colab, jax is preinstalled but the CUDA build differs.
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=False)

# Make the repo importable whether the notebook is opened from the repo root or Colab.
REPO = 'https://github.com/Mxgisha/chaotic-dynamical-systems-Research-tufts.git'
if not os.path.isdir('lorenz63'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO], check=False)
    os.chdir(os.path.basename(REPO)[:-4])
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

In [ ]:
import json
from pathlib import Path

from lorenz63.backend import BACKEND
from lorenz63.diagnostics import available
from lorenz63.report import is_blank_png
from scripts.run_sweep import load_config, run_point

print('array backend:', BACKEND)
print('diagnostics:', available())

In [ ]:
SMOKE = True          # True = one point, small config, a few minutes on CPU
BACKEND = 'numpy'     # 'numpy' (any machine) or 'jax' (needs requirements-jax.txt)
OUT = Path('runs')

cfg = load_config('configs/sweep.json')
diag_options = dict(cfg['diagnostics'])
if SMOKE:
    s = cfg['smoke']
    cfg['data'].update({k: s[k] for k in
                        ('train_steps', 'eval_steps', 'burn_in', 'n_short', 'short_steps')})
    cfg['train']['epochs'] = s['epochs']
    diag_options.update(s['diagnostics'])
    cfg['sweep'] = cfg['sweep'][:1]

all_results = {}
for entry in cfg['sweep']:
    name = f"test_{entry['noise_scale']}_{entry['horizon']}"
    print(f'=== {name} ===')
    all_results[name] = run_point(cfg, entry, OUT,
                                  backend=BACKEND, diag_options=diag_options) 

In [ ]:
# Ground-truth vs emulator ratios for every measurable metric.
rows = []
for point, results in all_results.items():
    for r in results:
        for metric, ratio in r.ratios().items():
            rows.append({'point': point, 'diagnostic': r.name,
                         'metric': metric, 'emu_over_gt': round(ratio, 4),
                         'gt': r.gt.get(metric), 'emu': r.emu.get(metric),
                         'gt_gt_control': (r.control_gt_gt or {}).get(metric)})
print(f'{len(rows)} metrics across {len(all_results)} sweep points')

try:
    import pandas as pd  # optional, for a nicer table
    display(pd.DataFrame(rows))
except ImportError:
    print(f"{'diagnostic':24s} {'metric':24s} {'gt':>12s} {'emu':>12s} {'ratio':>9s}")
    for row in rows:
        gt = row['gt'] if row['gt'] is not None else float('nan')
        emu = row['emu'] if row['emu'] is not None else float('nan')
        print(f"{row['diagnostic']:24s} {row['metric']:24s} {gt:12.4g} {emu:12.4g} {row['emu_over_gt']:9.3f}")

In [ ]:
# Artifact audit: every PNG must contain visible content.
pngs = sorted(OUT.rglob('*.png'))
blanks = [str(p) for p in pngs if is_blank_png(p)]
print(f'{len(pngs)} figures written, {len(blanks)} blank')
for p in blanks:
    print('  BLANK:', p)
assert not blanks, 'blank artifacts were produced'

In [ ]:
# Show a couple of diagnostics inline.
from IPython.display import Image, display

point_dir = next(OUT.glob('test_*'))
for name in ('lyapunov_spectrum', 'symbolic_dynamics', 'correlation_dimension'):
    p = point_dir / f'{name}.png'
    if p.exists():
        print(name)
        display(Image(filename=str(p)))

In [ ]:
# Full results, including the null controls, for one point.
payload = json.loads((point_dir / 'results.json').read_text())
for r in payload:
    print('=' * 70)
    print(r['name'])
    print('  gt  :', {k: v for k, v in r['gt'].items() if not k.startswith('_')})
    print('  emu :', {k: v for k, v in r['emu'].items() if not k.startswith('_')})